# Update: extend to 70 Ma and add 0.5 Ma interpolation

This notebook was programmatically updated to:
1. **Extend** any time/age limits that looked like 50–70 Ma caps to **70 Ma**.
2. **Append** a new section that **interpolates ocean areas** sampled at **5 Myr** intervals to **0.5 Myr** intervals.

> If anything looks off, search for parameter cells that define end age / max age and adjust manually.


In [ ]:
# --- Normalize age column name from the known CSV header ---
# The CSV produced earlier uses: Time (Ma), Pacific (million km²), Atlantic (million km²), ...
# Rename 'Time (Ma)' to the internal standard 'Age_Ma' so downstream code works consistently.
import pandas as pd

if 'ocean_areas_df' in globals():
    if 'Time (Ma)' in ocean_areas_df.columns:
        ocean_areas_df = ocean_areas_df.rename(columns={'Time (Ma)': 'Age_Ma'})
        print("Renamed 'Time (Ma)' -> 'Age_Ma' in ocean_areas_df")
else:
    # If reading from CSV later, the interpolation cell will also handle this,
    # but we keep this normalization here in case the DataFrame already exists.
    pass


In [ ]:
# --- Fixed data source for 5 Myr ocean areas ---
# Using the path provided by the user:
OCEAN_AREAS_CSV = "ocean_basin_area_outputs/ocean_basin_areas_0_70Ma_million_km2.csv"
print(f"Using OCEAN_AREAS_CSV: {OCEAN_AREAS_CSV}")


# Ocean Basins via Reconstructable Boundaries


## Concepts & Assumptions
**Spherical everything.** We treat polygons, masks and areas on the *unit sphere*, not on a projected plane.
This avoids antimeridian artefacts and area distortions.

**Dynamic land.** Continental polygons (Müller2019) are reconstructed at time *t* using each feature’s
reconstruction plate ID and the global rotation model, producing a time‑dependent **land mask**.

**Curtains (barriers).** Basin boundaries are represented as *thin blocking bands*:
- **Meridian bands** (constant longitude ± half‑width), implemented with modular longitudes so they cross the antimeridian cleanly.
- **Great‑circle segments** (geodesics), sampled densely and thickened into bands.
These do not remove ocean; they only prevent connectivity across them.

**Flood‑fill labeling on a periodic grid.** From a handful of **seeds** in each basin, a breadth‑first search
labels water cells connected **without crossing land or curtains**. Longitude wraps (x+1 at 179.5° becomes −180.5°).

**Arctic excluded.** Latitudes > 66.5°N are masked out. **Antarctica is the southern boundary**.

**Numerics.**
- Lon “distance” uses \( \Delta\lambda = ((\lambda - \lambda_0 + 180^\circ) \bmod 360^\circ) - 180^\circ \).
- Spherical cell area for a latitude band \([\phi_1,\phi_2]\) and width \(\Delta\lambda\):
  \[ A = R^2\, \Delta\lambda\, (\sin\phi_2 - \sin\phi_1) .\]
- Flood‑fill is \(\mathcal{O}(N)\) over ocean cells. Point‑in‑polygon dominates runtime; resolution controls cost.


## Imports & Config
**What’s configured here**
- **Grid**: 1°×1° lon/lat; increase to 0.5° for crisper coastlines (slower).
- **Times**: 0–70 Ma at 5 Ma steps.
- **Files**: Müller2019 continental polygons (already plate‑partitioned) and Natural Earth marine polygons (today only).
- **Curtain widths**: choose half‑widths (°) to guarantee closure at joins without over‑masking.
- **Output**: now saved under **`ocean_basin_area_outputs`**.


In [ ]:

# ---- imports & config ----
import os, math, csv, zipfile
import numpy as np
import matplotlib.pyplot as plt
try:
    import pygplates
except Exception as e:
    raise RuntimeError("pygplates is required (1.0 ok). Import error: %r" % (e,))
try:
    from gplately import PlateModelManager
except Exception:
    PlateModelManager = None

CONT_POLY_FILE = "muller2019/ContPolygons/Global_PresentDay_ContPolygons_2019_v1.shp"  # with .dbf/.shx
MODEL_NAME = "Muller2019"
DLON, DLAT = 1.0, 1.0
TIMES = list(range(0, 71, 5))  # 0..70 Ma
ARCTIC_CUTOFF_LAT = 66.5
OUT_DIR = "ocean_basin_area_outputs"
CURTAIN_HALFWIDTH_DEG = 1.5
TAS_CURTAIN_HALFWIDTH = 2.0  # slightly wider to ensure closure
NE_ZIP = "ne_10m_geography_marine_polys.zip"

# Tasmania curtain latitude band (only southern-ocean portion)
TAS_LAT_MIN = -80.0
TAS_LAT_MAX = -30.0
JOIN_LAT = -30.0  # WP↔Tas join latitude

os.makedirs(OUT_DIR, exist_ok=True)
if not os.path.exists(CONT_POLY_FILE):
    raise FileNotFoundError(f"Continental polygons file not found: {CONT_POLY_FILE}")
if not os.path.exists(NE_ZIP):
    raise FileNotFoundError(f"Natural Earth zip not found: {NE_ZIP}")

# unzip Natural Earth
ne_dir = os.path.join(OUT_DIR, "natural_earth"); os.makedirs(ne_dir, exist_ok=True)
with zipfile.ZipFile(NE_ZIP, 'r') as zf: zf.extractall(ne_dir)
ne_shp = None
for root, _, files in os.walk(ne_dir):
    for f in files:
        if f.lower().endswith(".shp"):
            ne_shp = os.path.join(root, f); break
    if ne_shp: break
if ne_shp is None:
    raise FileNotFoundError("No .shp found inside Natural Earth zip")
print("Natural Earth shapefile:", os.path.basename(ne_shp))


## Helpers
**Key routines**
- `poly_contains`: spherical point‑in‑polygon. Tries `pygplates` native; falls back to an **angle‑sum** test on the unit sphere.
  For polygon vertices \\(\\{\\mathbf v_i\\}\\) and test point \\(\\mathbf p\\), the signed angle between consecutive arcs from
  \\(\\mathbf p\\) to \\(\\mathbf v_i\\) accumulates to \\(\\pm 2\\pi\\) inside and ~0 outside.
- `node_areas`: per‑node spherical area using
  \\( A = R^2 \\Delta\\lambda (\\sin\\phi_2 - \\sin\\phi_1) \\).
- `ensure_rotation_model`, `get_plate_id_safe`: normalize inputs and extract plate IDs robustly.

**Dateline safety**
We never compare raw longitudes; we use modular deltas so \\(+179.9^\\circ\\) and \\(-179.9^\\circ\\) are “1/5° apart”.


In [ ]:

# ---- helpers (pygplates compat, grid, areas) ----
def poly_contains(poly, point):
    if hasattr(poly, 'is_point_in_polygon'):
        try:
            return bool(poly.is_point_in_polygon(point))
        except Exception:
            pass
    if hasattr(pygplates.GeometryOnSphere, 'is_point_in_polygon'):
        try:
            return bool(pygplates.GeometryOnSphere.is_point_in_polygon(point, poly))
        except Exception:
            pass
    # fallback spherical angle sum
    lat, lon = point.to_lat_lon()
    lat, lon = math.radians(lat), math.radians(lon)
    cp = np.array([math.cos(lat)*math.cos(lon), math.cos(lat)*math.sin(lon), math.sin(lat)], float)
    verts = []
    for v in poly.get_points():
        la, lo = v.to_lat_lon(); la, lo = math.radians(la), math.radians(lo)
        verts.append(np.array([math.cos(la)*math.cos(lo), math.cos(la)*math.sin(lo), math.sin(la)], float))
    if len(verts) < 3: return False
    total = 0.0
    for i in range(len(verts)):
        a = verts[i]; b = verts[(i+1)%len(verts)]
        num = float(np.dot(cp, np.cross(a, b)))
        den = float(np.dot(a, b) - np.dot(a, cp)*np.dot(b, cp))
        total += math.atan2(num, den)
    return abs(total) > math.pi

def get_plate_id_safe(feat):
    try:
        pid = feat.get_reconstruction_plate_id()
        if pid is not None: return int(pid)
    except Exception:
        pass
    for name in ("PLATEID1","PlateID","plate_id","ReconstructionPlateId","recon_plate_id"):
        try:
            props = feat.get_properties_by_name(name)
            if props: return int(props[0].get_value())
        except Exception:
            pass
    return 0

def ensure_rotation_model(rot_obj):
    if isinstance(rot_obj, pygplates.RotationModel):
        return rot_obj
    for attr in ('to_pygplates','as_rotation_model','rotation_model','to_rotation_model'):
        if hasattr(rot_obj, attr):
            v = getattr(rot_obj, attr); v = v() if callable(v) else v
            if isinstance(v, pygplates.RotationModel): return v
    if isinstance(rot_obj, (str,bytes)):
        return pygplates.RotationModel(rot_obj)
    try:
        import collections.abc
        if isinstance(rot_obj, collections.abc.Iterable):
            files = []
            for it in rot_obj:
                if isinstance(it, (str,bytes)): files.append(str(it))
                elif hasattr(it,'path'):
                    try: files.append(str(it.path))
                    except Exception: pass
            if files: return pygplates.RotationModel(files)
    except Exception:
        pass
    raise TypeError(f"Cannot normalize rotation model from {type(rot_obj)}")

R_EARTH_KM = 6371.0
def lonlat_grid(dlon=1.0, dlat=1.0):
    lons = np.arange(-180.0, 180.0 + 1e-9, dlon, dtype=float)
    lats = np.arange(-90.0,   90.0 + 1e-9, dlat, dtype=float)
    return lons, lats

def node_areas(lons, lats):
    dlon = np.deg2rad(float(np.mean(np.diff(lons))))
    dlat = np.deg2rad(float(np.mean(np.diff(lats))))
    lat  = np.deg2rad(lats.astype(float))
    lat_hi = np.clip(lat + 0.5*dlat, -np.pi/2, np.pi/2)
    lat_lo = np.clip(lat - 0.5*dlat, -np.pi/2, np.pi/2)
    band = (np.sin(lat_hi) - np.sin(lat_lo))[:, None]
    return (R_EARTH_KM**2) * dlon * band * np.ones((1, lons.size), float)


## Datasets
- **Rotation model**: Müller2019 via GPlately PMM → `pygplates.RotationModel`.
- **Continental polygons**: reconstructed per plate → **land mask at t**.
- **Natural Earth oceans**: used only to *infer present‑day* meridians for initialization/sanity checks.


In [ ]:

# ---- datasets (PMM, continents, NE) ----
if PlateModelManager is None:
    raise RuntimeError("GPlately required to obtain Müller2019 rotation model")
model = PlateModelManager().get_model(MODEL_NAME)
rotation_model_pg = ensure_rotation_model(model.get_rotation_model())
print("Rotation model ready (Müller2019)")

cont_fc = pygplates.FeatureCollection(CONT_POLY_FILE)
cont_features = list(cont_fc)
print("Continental polygons:", len(cont_features))
if not cont_features: raise RuntimeError("No features in continental polygons")

ne_fc = pygplates.FeatureCollection(ne_shp)
ne_features = list(ne_fc)
print("Natural Earth marine polys:", len(ne_features))


## Ne Inference
We scan longitudes and watch **ocean class changes** (Pacific→Indian, Atlantic→Indian) at a handful of latitudes.
The median cut provides a robust present‑day meridian guess. These values are only fallbacks for anchors —
time evolution comes from reconstructing the anchor longitudes with plates.


In [ ]:

# ---- Natural Earth ocean tags & meridian inference ----
def feat_name_lower(feat):
    for nm in ("name","name_en","NAME","Name","featurecla"):
        try:
            props = feat.get_properties_by_name(nm)
            if props: return str(props[0].get_value()).strip().lower()
        except Exception:
            pass
    return ""

NE_NAMES = {"pacific ocean","atlantic ocean","indian ocean","southern ocean","arctic ocean"}
ne_ocean_feats = [f for f in ne_features if feat_name_lower(f) in NE_NAMES]

def classify_ocean_NE(lon, lat):
    P = pygplates.PointOnSphere(lat, lon)
    for f in ne_ocean_feats:
        g = f.get_geometry()
        if g is None: continue
        if isinstance(g, pygplates.PolygonOnSphere):
            polys = [g]
        elif hasattr(pygplates,'MultiPolygonOnSphere') and isinstance(g, pygplates.MultiPolygonOnSphere):
            polys = list(g)
        else:
            polys = []
        for poly in polys:
            try:
                if poly_contains(poly, P): return feat_name_lower(f)
            except Exception:
                pass
    return None

def scan_boundary_meridian(lats_to_sample, west_name, east_name, lon_range):
    lons = np.linspace(lon_range[0], lon_range[1], 721)  # ~0.5°
    cuts = []
    for la in lats_to_sample:
        prev = None
        for lo in lons:
            nm = classify_ocean_NE(lo, la)
            if nm != prev and prev is not None:
                if prev == west_name and nm == east_name:
                    cuts.append(lo); break
            prev = nm
    if not cuts: return None, []
    return float(np.median(cuts)), cuts

# Inferred present-day meridians as fallbacks (not directly used in seed gating)
L_AF_meridian, _ = scan_boundary_meridian([-40,-30,-20,-10,0,5], "atlantic ocean", "indian ocean", (0,60))
L_WP_meridian, _ = scan_boundary_meridian([20,10,5,0,-10,-20], "indian ocean", "pacific ocean", (110,160))
if L_AF_meridian is None: L_AF_meridian = 20.0
if L_WP_meridian is None: L_WP_meridian = 136.0
print("Present meridians (inferred): Africa", L_AF_meridian, "| WestPacific", L_WP_meridian)


## Static/Anchors
**Static polygons** resolve the **plate ID** at arbitrary anchor coordinates so anchors move correctly with plates.

**Anchors used**
- **Africa meridian** near Agulhas (plate 701).
- **West‑of‑Philippines meridian** around ~136°E with a **time‑varying plate** at ~15°N.
- **Tasmania meridian** aligned to reconstructed Tasmania longitude (plate 801) between −80° and −30°.
- **Americas closure**: two geodesic segments from Central America down the west flank of SA to prevent Pac↔Atl mixing pre‑Isthmus.
- **Drake Passage**: SA southern tip (plate 201) ↔ Antarctic Peninsula.

**Seed gating**
Pacific seeds are kept **east of WP**; Indian seeds **west of WP**, with
\( \text{east} := ((\lambda - \lambda_{WP} + 360) \bmod 360) < 180 \).


In [ ]:

# ---- static polygons, anchors, meridians ----
def _to_feature_seq_simple(obj):
    if isinstance(obj, pygplates.Feature): return [obj]
    if isinstance(obj, pygplates.FeatureCollection): return list(obj)
    if isinstance(obj, (str,bytes)): return list(pygplates.FeatureCollection(obj))
    for attr in ('to_feature_collection','as_feature_collection','to_pygplates','feature_collection','features','paths','path','files','file'):
        if hasattr(obj, attr):
            v = getattr(obj, attr); v = v() if callable(v) else v
            return _to_feature_seq_simple(v)
    try:
        import collections.abc
        if isinstance(obj, collections.abc.Iterable):
            out=[]
            for it in obj:
                try: out.extend(_to_feature_seq_simple(it))
                except Exception: pass
            if out: return out
    except Exception: pass
    raise TypeError(f"Cannot convert {type(obj)} to pygplates.Feature sequence")

def load_static_polygons_features(model):
    names = []
    try: names = list(getattr(model,'layers',{}).keys())
    except Exception: names = list(getattr(model,'layer_dict',{}).keys())
    preferred = ['StaticPolygons','Global_EarthByte_GPlates_StaticPolygons_v1','GlobalStaticPolygons']
    for nm in preferred + names:
        try:
            lyr = model.get_layer(nm)
            feats = _to_feature_seq_simple(lyr)
            if feats: return feats, nm
        except Exception:
            pass
    raise RuntimeError("StaticPolygons layer not found; available: " + ", ".join(names))

static_features, static_name = load_static_polygons_features(model)

def resolve_plate_id_from_static(lon, lat):
    P = pygplates.PointOnSphere(lat, lon)
    for feat in static_features:
        g = feat.get_geometry()
        if g is None: continue
        if isinstance(g, pygplates.PolygonOnSphere):
            polys=[g]
        elif hasattr(pygplates,'MultiPolygonOnSphere') and isinstance(g, pygplates.MultiPolygonOnSphere):
            polys=list(g)
        else:
            polys=[]
        for poly in polys:
            if poly_contains(poly, P): return get_plate_id_safe(feat)
    return 0

def min_lat_point_for_plate(plate_id):
    best=None
    for f in cont_features:
        if get_plate_id_safe(f)!=plate_id: continue
        g=f.get_geometry()
        if g is None: continue
        pts=[]
        if isinstance(g, pygplates.PolygonOnSphere): pts=g.get_points()
        elif hasattr(pygplates,'MultiPolygonOnSphere') and isinstance(g, pygplates.MultiPolygonOnSphere):
            for sub in g: pts.extend(sub.get_points())
        for p in pts:
            la, lo = p.to_lat_lon()
            if (best is None) or (la < best[1]):
                best = (float(lo), float(la))
    return best

# Anchors
ANTPEN_ANCHOR0 = (-58.5, -63.5)  # Antarctic Peninsula
AU_PLATE = 801
SA_PLATE = 201
AF_PLATE = 701

tas_tip_present = min_lat_point_for_plate(AU_PLATE)
sa_tip_present  = min_lat_point_for_plate(SA_PLATE)
if not sa_tip_present or not tas_tip_present:
    raise RuntimeError("Could not locate SA or Tasmania tips from continental polygons")

ANTPEN_PID = resolve_plate_id_from_static(*ANTPEN_ANCHOR0)

# WP anchor (present-day) and plate
L_WP0 = float(L_WP_meridian)
WP_ANCHOR0 = (L_WP0, 15.0)
wp_pid = resolve_plate_id_from_static(*WP_ANCHOR0)


## Meridional boundary builders
**Meridian bands**
A meridian at longitude \(\lambda_c\) becomes a band where \(|\Delta\lambda(\lambda,\lambda_c)| \le w\) and
\(\phi_{\min} \le \phi \le \phi_{\max}\). This is antimeridian‑safe.

**Geodesic polylines**
We connect reconstructed anchors by **great‑circle** interpolation (SLERP on the sphere), then thicken to a band by
marking all pixels within a small geodesic distance (approximated by Euclidean distance in \(\Delta\lambda\cos\phi,\Delta\phi\)).
A small dilation step reduces pinholes.

**Join near Tasmania**
We connect the WP meridian to the Tasmania meridian with a short great‑circle segment centered near **−30°**.
A band half‑width of ~1.5° ensures a watertight join without overmasking.

**Periodic flood‑fill**
BFS grows labels through ocean cells; longitude wraps modulo the grid width so basins spanning the antimeridian remain connected.


In [ ]:

# ---- curtain builders ----
def recon_lon_of_point(lon, lat, plate_id, t_ma):
    P0 = pygplates.PointOnSphere(lat, lon)
    R = rotation_model_pg.get_rotation(float(t_ma), int(plate_id))
    P = R * P0
    la, lo = P.to_lat_lon()
    return float(lo)

def recon_point(lon, lat, plate_id, t_ma):
    P0 = pygplates.PointOnSphere(lat, lon)
    R = rotation_model_pg.get_rotation(float(t_ma), int(plate_id))
    P  = R * P0
    la, lo = P.to_lat_lon()
    return (float(lo), float(la))

def build_meridian_mask(lons, lats, center_lon, lat_min=-80, lat_max=75, halfwidth_deg=1.5):
    LX, LY = np.meshgrid(lons, lats)
    band = (LY >= lat_min) & (LY <= lat_max)
    dlon = (LX - center_lon + 180.0) % 360.0 - 180.0
    return band & (np.abs(dlon) <= halfwidth_deg)

def geodesic_polyline(lonlat_a, lonlat_b, npts=700):
    la1, lo1 = map(math.radians, (lonlat_a[1], lonlat_a[0]))
    la2, lo2 = map(math.radians, (lonlat_b[1], lonlat_b[0]))
    a = np.array([math.cos(la1)*math.cos(lo1), math.cos(la1)*math.sin(lo1), math.sin(la1)], float)
    b = np.array([math.cos(la2)*math.cos(lo2), math.cos(la2)*math.sin(lo2), math.sin(la2)], float)
    dot = max(-1.0, min(1.0, float(np.dot(a,b))))
    ang = math.acos(dot)
    if ang == 0: return [lonlat_a]*npts
    out = []
    for k in range(npts):
        t = k/(npts-1)
        s1 = math.sin((1-t)*ang)/math.sin(ang); s2 = math.sin(t*ang)/math.sin(ang)
        v = s1*a + s2*b; v /= np.linalg.norm(v)
        la = math.degrees(math.asin(v[2])); lo = math.degrees(math.atan2(v[1], v[0]))
        out.append((lo, la))
    return out

def rasterize_polyline_band(lons, lats, polyline_lonlat, halfwidth_deg):
    LX, LY = np.meshgrid(lons, lats)
    mask = np.zeros_like(LX, dtype=bool)
    for (lo, la) in polyline_lonlat:
        dlon = (LX - lo + 180.0) % 360.0 - 180.0
        dlat = LY - la
        mask |= (np.hypot(dlon*np.cos(np.deg2rad(la)), dlat) <= halfwidth_deg)
    m = mask.copy()
    m[:,1:] |= mask[:,:-1]; m[:,:-1] |= mask[:,1:]
    m[1:,:] |= mask[:-1,:]; m[:-1,:] |= mask[1:,:]
    return m

def is_east_of(lon, ref_lon):
    return ((lon - ref_lon + 360.0) % 360.0) < 180.0

PAC, ATL, IND = 1, 2, 3

# Base seeds (we will filter per-time using WP longitude)
SEEDS_PAC_BASE = [(-160.0, 0.0), (150.0, 15.0), (170.0, -30.0), (135.0, 15.0)]
SEEDS_ATL_BASE = [(-30.0, 0.0), (-10.0, 20.0), (-40.0, -40.0)]
SEEDS_IND_BASE = [(80.0, -20.0), (60.0, 10.0), (100.0, -15.0)]


## Main Loop
Per time step:
1. Reconstruct continental polygons → **land mask** (spherical point‑in‑polygon).
2. **Ocean mask** = not land; remove Arctic (>66.5°N).
3. Build **curtains**: Africa meridian, WP meridian (down to −30°), Tasmania meridian (−80→−30°), WP↔Tas join, Americas, Drake.
4. **Gate seeds** by WP meridian and run **periodic flood‑fill** to produce labels (Pac=1, Atl=2, Ind=3).
5. Sum **spherical node areas** per label; render map PNG.

**Checks**
- If two basins merge on a map, widen the local band (e.g., `CURTAIN_HALFWIDTH_DEG=2.0`) or adjust the anchors slightly.
- If a pocket remains unlabeled, add a seed inside that pocket (rare at 1° grid).


In [ ]:

# ---- main loop + collect time series ----
lons, lats = lonlat_grid(DLON, DLAT)
areas = node_areas(lons, lats)
arctic = (lats >= ARCTIC_CUTOFF_LAT)

def wp_lat_cap(t):
    if t <= 10: return 12.0
    if t <= 30: return 20.0
    return 35.0

WP_LAT_MIN = -30.0  # join with Tasmania meridian

T_list, Pac_list, Atl_list, Ind_list = [], [], [], []

for t in TIMES:
    print(f"Time {t} Ma …")

    # Land mask via reconstructed continents (Antarctica acts as south boundary; no global SO curtain)
    polys = []
    for feat in cont_features:
        g = feat.get_geometry()
        if g is None: continue
        pid = get_plate_id_safe(feat)
        R = rotation_model_pg.get_rotation(float(t), int(pid))
        if isinstance(g, pygplates.PolygonOnSphere):
            pts = [R*p for p in g.get_points()] if t!=0 else list(g.get_points())
            polys.append(pygplates.PolygonOnSphere(pts))
        elif hasattr(pygplates,'MultiPolygonOnSphere') and isinstance(g, pygplates.MultiPolygonOnSphere):
            for sub in g:
                pts = [R*p for p in sub.get_points()] if t!=0 else list(sub.get_points())
                polys.append(pygplates.PolygonOnSphere(pts))

    LX, LY = np.meshgrid(lons, lats)
    pts = [pygplates.PointOnSphere(float(la), float(lo)) for la,lo in zip(LY.ravel(), LX.ravel())]
    inside = np.zeros(LX.size, dtype=bool)
    for poly in polys:
        for i, P in enumerate(pts):
            if not inside[i] and poly_contains(poly, P):
                inside[i] = True
        if inside.all(): break
    land_mask = inside.reshape(lats.size, lons.size)

    ocean_mask = ~land_mask
    ocean_mask[arctic, :] = False  # exclude Arctic

    # Africa (Atlantic–Indian) meridian — anchored to Africa southern tip longitude (reconstructed)
    af_tip = min_lat_point_for_plate(701)
    af_pid = resolve_plate_id_from_static(*af_tip) if af_tip else 701
    L_af_t = recon_lon_of_point(af_tip[0], af_tip[1], af_pid, t) if af_tip else 20.0
    africa_mer = build_meridian_mask(lons, lats, L_af_t, lat_min=-80, lat_max=75, halfwidth_deg=CURTAIN_HALFWIDTH_DEG)

    # WP meridian — down to JOIN_LAT
    L_wp_t = recon_lon_of_point(WP_ANCHOR0[0], WP_ANCHOR0[1], resolve_plate_id_from_static(*WP_ANCHOR0), t)
    wp_mer  = build_meridian_mask(lons, lats, L_wp_t,  lat_min=WP_LAT_MIN, lat_max=wp_lat_cap(t), halfwidth_deg=CURTAIN_HALFWIDTH_DEG)

    # Tasmania‑aligned meridian
    tas_lon_t = recon_lon_of_point(tas_tip_present[0], tas_tip_present[1], 801, t)
    tas_mask  = build_meridian_mask(lons, lats, center_lon=tas_lon_t, 
                                    lat_min=TAS_LAT_MIN, lat_max=TAS_LAT_MAX, 
                                    halfwidth_deg=TAS_CURTAIN_HALFWIDTH)

    # Join WP meridian to Tasmania meridian along JOIN_LAT (−30°)
    def geodesic_polyline(lonlat_a, lonlat_b, npts=600):
        la1, lo1 = math.radians(lonlat_a[1]), math.radians(lonlat_a[0])
        la2, lo2 = math.radians(lonlat_b[1]), math.radians(lonlat_b[0])
        a = np.array([math.cos(la1)*math.cos(lo1), math.cos(la1)*math.sin(lo1), math.sin(la1)], float)
        b = np.array([math.cos(la2)*math.cos(lo2), math.cos(la2)*math.sin(lo2), math.sin(la2)], float)
        dot = max(-1.0, min(1.0, float(np.dot(a,b)))); ang = math.acos(dot)
        if ang == 0: return [lonlat_a]*npts
        out = []
        for k in range(npts):
            t_ = k/(npts-1)
            s1 = math.sin((1-t_)*ang)/math.sin(ang); s2 = math.sin(t_*ang)/math.sin(ang)
            v = s1*a + s2*b; v /= np.linalg.norm(v)
            la = math.degrees(math.asin(v[2])); lo = math.degrees(math.atan2(v[1], v[0]))
            out.append((lo, la))
        return out
    join_line = geodesic_polyline((L_wp_t, JOIN_LAT), (tas_lon_t, JOIN_LAT), npts=600)
    def rasterize_polyline_band(lons, lats, polyline_lonlat, halfwidth_deg):
        LX, LY = np.meshgrid(lons, lats)
        mask = np.zeros_like(LX, dtype=bool)
        for (lo, la) in polyline_lonlat:
            dlon = (LX - lo + 180.0) % 360.0 - 180.0
            dlat = LY - la
            mask |= (np.hypot(dlon*np.cos(np.deg2rad(la)), dlat) <= halfwidth_deg)
        m = mask.copy()
        m[:,1:] |= mask[:,:-1]; m[:,:-1] |= mask[:,1:]
        m[1:,:] |= mask[:-1,:]; m[:-1,:] |= mask[1:,:]
        return m
    join_mask = rasterize_polyline_band(lons, lats, join_line, CURTAIN_HALFWIDTH_DEG)

    # Americas extended curtain (NA->SA1->SA2) + Drake
    def recon_point(lon, lat, plate_id, t_ma):
        P0 = pygplates.PointOnSphere(lat, lon); R = rotation_model_pg.get_rotation(float(t_ma), int(plate_id)); P = R*P0
        la, lo = P.to_lat_lon(); return (float(lo), float(la))
    NA_ANCHOR0 = (-95.0, 16.0); SA_ANCHOR1 = (-78.0, 8.0); SA_ANCHOR2 = (-80.0, -5.0)
    NA_PID = resolve_plate_id_from_static(*NA_ANCHOR0); SA1_PID = resolve_plate_id_from_static(*SA_ANCHOR1); SA2_PID = resolve_plate_id_from_static(*SA_ANCHOR2)
    na_t  = recon_point(NA_ANCHOR0[0],  NA_ANCHOR0[1],  NA_PID,  t)
    sa1_t = recon_point(SA_ANCHOR1[0], SA_ANCHOR1[1], SA1_PID, t)
    sa2_t = recon_point(SA_ANCHOR2[0], SA_ANCHOR2[1], SA2_PID, t)
    def geodesic_polyline_long(a,b,npts=600):
        la1, lo1 = math.radians(a[1]), math.radians(a[0])
        la2, lo2 = math.radians(b[1]), math.radians(b[0])
        A = np.array([math.cos(la1)*math.cos(lo1), math.cos(la1)*math.sin(lo1), math.sin(la1)], float)
        B = np.array([math.cos(la2)*math.cos(lo2), math.cos(la2)*math.sin(lo2), math.sin(la2)], float)
        dot = max(-1.0, min(1.0, float(np.dot(A,B)))); ang = math.acos(dot)
        if ang == 0: return [a]*npts
        out=[]
        for k in range(npts):
            t_ = k/(npts-1)
            s1 = math.sin((1-t_)*ang)/math.sin(ang); s2 = math.sin(t_*ang)/math.sin(ang)
            v = s1*A + s2*B; v /= np.linalg.norm(v)
            la = math.degrees(math.asin(v[2])); lo = math.degrees(math.atan2(v[1], v[0]))
            out.append((lo, la))
        return out
    amer_line1 = geodesic_polyline_long(na_t,  sa1_t, npts=600)
    amer_line2 = geodesic_polyline_long(sa1_t, sa2_t, npts=600)
    amer_mask  = (rasterize_polyline_band(lons, lats, amer_line1, CURTAIN_HALFWIDTH_DEG) |
                  rasterize_polyline_band(lons, lats, amer_line2, CURTAIN_HALFWIDTH_DEG))

    sa_tip_pid = resolve_plate_id_from_static(*sa_tip_present) if sa_tip_present else 201
    antpen_t  = recon_point(-58.5, -63.5, ANTPEN_PID, t)
    def geodesic_polyline_short(a,b,npts=700):
        la1, lo1 = math.radians(a[1]), math.radians(a[0])
        la2, lo2 = math.radians(b[1]), math.radians(b[0])
        A = np.array([math.cos(la1)*math.cos(lo1), math.cos(la1)*math.sin(lo1), math.sin(la1)], float)
        B = np.array([math.cos(la2)*math.cos(lo2), math.cos(la2)*math.sin(lo2), math.sin(la2)], float)
        dot = max(-1.0, min(1.0, float(np.dot(A,B)))); ang = math.acos(dot)
        if ang == 0: return [a]*npts
        out=[]
        for k in range(npts):
            t_ = k/(npts-1)
            s1 = math.sin((1-t_)*ang)/math.sin(ang); s2 = math.sin(t_*ang)/math.sin(ang)
            v = s1*A + s2*B; v /= np.linalg.norm(v)
            la = math.degrees(math.asin(v[2])); lo = math.degrees(math.atan2(v[1], v[0]))
            out.append((lo, la))
        return out
    sa_tip_t  = recon_point(sa_tip_present[0], sa_tip_present[1], sa_tip_pid, t)
    drake_line = geodesic_polyline_short(sa_tip_t, antpen_t, npts=700)
    drake_mask = rasterize_polyline_band(lons, lats, drake_line, CURTAIN_HALFWIDTH_DEG)

    curtain_mask = africa_mer | wp_mer | tas_mask | join_mask | amer_mask | drake_mask

    # prevent flood across curtains
    ocean_mask &= ~curtain_mask

    # time-aware seeds relative to WP meridian
    def filtered_seeds(L_wp):
        pac, ind = [], []
        for (lo, la) in SEEDS_PAC_BASE:
            if is_east_of(lo, L_wp): pac.append((lo, la))
        for (lo, la) in SEEDS_IND_BASE:
            if not is_east_of(lo, L_wp): ind.append((lo, la))
        return pac, ind
    pac_seeds, ind_seeds = filtered_seeds(L_wp_t)

    # flood fill (labels: 1 Pac, 2 Atl, 3 Ind)
    from collections import deque
    def lonlat_to_index(lons, lats, lon, lat):
        ix = int(np.round((lon - lons[0]) / (lons[1]-lons[0]))); ix = max(0, min(ix, lons.size-1))
        iy = int(np.round((lat - lats[0]) / (lats[1]-lats[0]))); iy = max(0, min(iy, lats.size-1))
        return ix, iy
    def flood_fill_labels(ocean_mask, barrier_mask, lons, lats, pac_seeds, atl_seeds, ind_seeds):
        ny, nx = ocean_mask.shape
        labels = np.zeros((ny, nx), dtype=np.uint8)
        blocked = (~ocean_mask) | (barrier_mask)
        def push_seed_list(seed_list, lab):
            q = deque()
            for (lo, la) in seed_list:
                ix, iy = lonlat_to_index(lons, lats, lo, la)
                if not blocked[iy, ix] and labels[iy, ix] == 0:
                    labels[iy, ix] = lab; q.append((ix, iy))
            while q:
                x, y = q.popleft()
                for dx, dy in ((1,0),(-1,0),(0,1),(0,-1)):
                    xx = (x + dx) % nx; yy = y + dy
                    if yy < 0 or yy >= ny: continue
                    if blocked[yy, xx] or labels[yy, xx] != 0: continue
                    labels[yy, xx] = lab; q.append((xx, yy))
            return labels
        labels = push_seed_list(pac_seeds, 1)
        labels = push_seed_list(SEEDS_ATL_BASE, 2)
        labels = push_seed_list(ind_seeds, 3)
        return labels

    labels = flood_fill_labels(ocean_mask, curtain_mask, lons, lats, pac_seeds, SEEDS_ATL_BASE, ind_seeds)

    # areas
    A_pac = float(areas[labels==1].sum())
    A_atl = float(areas[labels==2].sum())
    A_ind = float(areas[labels==3].sum())

    T_list.append(t); Pac_list.append(A_pac); Atl_list.append(A_atl); Ind_list.append(A_ind)

    # map render
    from matplotlib.colors import ListedColormap, BoundaryNorm
    cmap = ListedColormap(['white','tab:blue','tab:orange','tab:green'])
    norm = BoundaryNorm([0,1,2,3,4], cmap.N)
    fig = plt.figure(figsize=(13,4))
    plt.title(f"Basins @ {t} Ma (blue=Pac, orange=Atl, green=Ind)")
    bg = labels.copy()
    plt.imshow(bg, origin='lower', extent=[lons.min(), lons.max(), lats.min(), lats.max()], cmap=cmap, norm=norm)
    # semi‑transparent land
    land_alpha = np.where(land_mask, 0.25, 0.0)
    plt.imshow(land_alpha, origin='lower', extent=[lons.min(), lons.max(), lats.min(), lats.max()], cmap='Greys', vmin=0, vmax=1, alpha=land_alpha)
    # overlays
    plt.plot([L_af_t]*2, [-80,75], 'k--', lw=1.2)
    plt.plot([L_wp_t]*2, [JOIN_LAT, wp_lat_cap(t)], 'k--', lw=1.2)
    plt.plot([tas_lon_t]*2, [TAS_LAT_MIN, TAS_LAT_MAX], 'k-', lw=1.8)
    xs=[p[0] for p in join_line]; ys=[p[1] for p in join_line]; plt.plot(xs, ys, 'k-', lw=1.6)
    # Americas & Drake
    def draw(pl, c='k', lw=1.6):
        xs=[p[0] for p in pl]; ys=[p[1] for p in pl]; plt.plot(xs, ys, c, lw=lw)
    draw(amer_line1); draw(amer_line2); draw(drake_line)
    plt.xlabel("Longitude"); plt.ylabel("Latitude"); plt.tight_layout()
    fig.savefig(os.path.join(OUT_DIR, f"basins_{t:02d}Ma.png"), dpi=150)
    plt.close(fig)

print("Done. PNGs in:", OUT_DIR)


## Xy Plot
Area time series (million km²) with present at right. Expect present‑day values roughly close to Wikipedia:
Pacific ~165.25, Atlantic ~107.0, Indian ~70.56 (differences arise from resolution and boundary definitions).


In [ ]:

# ---- XY plot of basin areas through time ----
import numpy as np, matplotlib.pyplot as plt, os

fig = plt.figure(figsize=(10,5))
x = np.array(T_list, float); order = np.argsort(x)
x = x[order]; pac = (np.array(Pac_list)[order])/1e6; atl=(np.array(Atl_list)[order])/1e6; ind=(np.array(Ind_list)[order])/1e6
plt.plot(x, pac, label="Pacific")
plt.plot(x, atl, label="Atlantic")
plt.plot(x, ind, label="Indian")
plt.gca().invert_xaxis()
plt.xlabel("Time (Ma)"); plt.ylabel("Area (million km²)")
plt.title("Ocean Basin Areas Through Time (0–70 Ma)")
plt.legend(); plt.tight_layout()
plt.show()
xy_path = os.path.join(OUT_DIR, "areas_timeseries.png")
fig.savefig(xy_path, dpi=150); plt.show()
print("Saved:", xy_path)


## Area table (0–70 Ma)
The table lists basin areas at each time step (million km²). CSVs are saved to `OUT_DIR`.

## Area Table
Creates a tidy table and writes CSVs to **`ocean_basin_area_outputs`**. The table includes a simple sum of the three basins
(million km²) for quick mass‑balance checks. The present‑day line is printed for a fast sanity check.


In [ ]:

# Build and show a per-time area table; also save CSVs (raw and million km²)
import pandas as pd, numpy as np, os

if not (len(T_list)==len(Pac_list)==len(Atl_list)==len(Ind_list)):
    raise RuntimeError("Area lists are inconsistent lengths. Run the main loop cell first.")

df = pd.DataFrame({
    "time_Ma": T_list,
    "Pacific_km2": Pac_list,
    "Atlantic_km2": Atl_list,
    "Indian_km2": Ind_list
}).sort_values("time_Ma", ascending=False).reset_index(drop=True)

df_tbl = pd.DataFrame({
    "Time (Ma)": df["time_Ma"],
    "Pacific (million km²)": df["Pacific_km2"] / 1e6,
    "Atlantic (million km²)": df["Atlantic_km2"] / 1e6,
    "Indian (million km²)": df["Indian_km2"] / 1e6,
})
df_tbl["Sum (million km²)"] = df_tbl[["Pacific (million km²)","Atlantic (million km²)","Indian (million km²)"]].sum(axis=1)

from IPython.display import display
display(df_tbl)

csv_raw = os.path.join(OUT_DIR, "ocean_basin_areas_0_70Ma.csv")
csv_m  = os.path.join(OUT_DIR, "ocean_basin_areas_0_70Ma_million_km2.csv")
df.to_csv(csv_raw, index=False)
df_tbl.to_csv(csv_m, index=False)
print("Saved:", csv_raw)
print("Saved:", csv_m)

# present-day sanity line
row0 = df_tbl[df_tbl["Time (Ma)"]==0]
if len(row0):
    r0 = row0.iloc[0]
    print("Present-day (0 Ma): Pac={:.1f} Atl={:.1f} Ind={:.1f} million km²".format(
        r0["Pacific (million km²)"], r0["Atlantic (million km²)"], r0["Indian (million km²)"]))


In [ ]:
# --- Configure data source for 5 Myr ocean areas ---
# If you already created `ocean_areas_df` above, this cell will skip.
# Otherwise, we try to auto-detect a CSV in the working directory with common keywords.
from pathlib import Path

try:
    ocean_areas_df  # type: ignore # noqa: F821
    print("Using existing DataFrame `ocean_areas_df`.")
except NameError:
    # Only set OCEAN_AREAS_CSV if not already set by the user
    if 'OCEAN_AREAS_CSV' in globals() and OCEAN_AREAS_CSV:
        print(f"Using OCEAN_AREAS_CSV from global: {OCEAN_AREAS_CSV}")
    else:
        # Heuristic search
        candidates = []
        for p in Path('.').glob('*.csv'):
            name = p.name.lower()
            if ('ocean' in name or 'basin' in name or 'sea' in name) and ('area' in name or 'areas' in name):
                candidates.append(p)
        if candidates:
            # Pick the largest file as a heuristic
            candidates = sorted(candidates, key=lambda p: p.stat().st_size, reverse=True)
            OCEAN_AREAS_CSV = str(candidates[0])
            print(f"Auto-detected CSV: {OCEAN_AREAS_CSV}")
        else:
            # Fallback: set this to your CSV path
            OCEAN_AREAS_CSV = ''  # e.g., 'data/ocean_basin_areas_5Myr.csv'
            if not OCEAN_AREAS_CSV:
                raise RuntimeError("Set OCEAN_AREAS_CSV to your 5 Myr CSV path, or define `ocean_areas_df`.")

In [ ]:
# --- Resolve data source for 5 Myr ocean areas (robust) ---
from pathlib import Path
import pandas as pd

print("Working directory:", Path.cwd().resolve())

# If a DataFrame was already defined, keep it
have_df = 'ocean_areas_df' in globals()

# If not, try to resolve an explicit OCEAN_AREAS_CSV or common filenames
if not have_df:
    tried = []
    # If user already set OCEAN_AREAS_CSV, try that first
    if 'OCEAN_AREAS_CSV' in globals() and OCEAN_AREAS_CSV:
        p = Path(OCEAN_AREAS_CSV)
        tried.append(str(p))
        if p.exists():
            print(f"Using existing OCEAN_AREAS_CSV: {p}")
            ocean_areas_df = pd.read_csv(p)
        else:
            print(f"Configured OCEAN_AREAS_CSV does not exist: {p}")
            have_df = False
    if not have_df:
        # Try canonical filenames (70 Ma preferred, then 65 Ma legacy)
        candidates = [
            Path("ocean_basin_area_outputs/ocean_basin_areas_0_70Ma_million_km2.csv"),
            Path("ocean_basin_area_outputs/ocean_basin_areas_0_65Ma_million_km2.csv"),
        ]
        for p in candidates:
            tried.append(str(p))
            if p.exists():
                print(f"Auto-detected CSV: {p}")
                ocean_areas_df = pd.read_csv(p)
                have_df = True
                break
    if not have_df:
        raise FileNotFoundError(
            "Could not find an input CSV for ocean-basin areas.\n"
            + "Checked the following paths (relative to working directory):\n  - "
            + "\n  - ".join(tried)
            + "\nPlease place the CSV at one of these paths or set OCEAN_AREAS_CSV to a valid path."
        )
else:
    print("Using existing DataFrame `ocean_areas_df`. Set `OCEAN_AREAS_CSV` to override.")


## Interpolate ocean areas from 5 Myr to 0.5 Myr (Spline)

This section upsamples a table of ocean-basin areas sampled at **5 Myr** steps to a **0.5 Myr** grid using a
**natural cubic spline** (via `scipy.interpolate.CubicSpline`).

**Assumptions**
- The 5 Myr table has a column `Age_Ma` and numeric columns for basin areas (e.g., `Pacific`, `Atlantic`, ...).
- Provide the data either as a DataFrame named `ocean_areas_df` **or** via a CSV path set in `OCEAN_AREAS_CSV`.
- Interpolation is done only within the source age domain (no extrapolation).


In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

try:
    from scipy.interpolate import CubicSpline
except Exception as e:
    raise ImportError("SciPy is required for spline interpolation. Please install scipy.") from e

# ---- Configure where to read the 5 Myr table ----
# Option A) Use an existing DataFrame named ocean_areas_df
# Option B) Or set OCEAN_AREAS_CSV to a CSV path
try:
    ocean_areas_df  # type: ignore # noqa
except NameError:
    OCEAN_AREAS_CSV = globals().get('OCEAN_AREAS_CSV', None)
    if OCEAN_AREAS_CSV is None:
        raise RuntimeError("Provide a 5 Myr table via a DataFrame named `ocean_areas_df` or set `OCEAN_AREAS_CSV`.")
    ocean_areas_df = pd.read_csv(OCEAN_AREAS_CSV)

# Ensure Age_Ma exists and sort by age (ascending)
import re
# ---- Robust age-column detection & normalization ----
cols = list(ocean_areas_df.columns)
# Normalize column names: lower, strip non-letters (so 'Age (Ma)' -> 'agema')
norm = {c: re.sub(r'[^a-z]', '', str(c).lower()) for c in cols}
# Priority exact matches, then prefix matches
preferred_keys = {'agema','agemyr','agemillionyears','timema','time'}
candidates = [c for c in cols if norm[c] in preferred_keys or norm[c]=='age']
if not candidates:
    candidates = [c for c in cols if norm[c].startswith('age') or norm[c].startswith('time')]
if not candidates:
    raise ValueError(f"Could not find an age column. Available columns: {cols}")
age_col = candidates[0]
print(f"Using age column: {age_col}")

# Ensure numeric age, drop NaNs, sort ascending, drop duplicate ages (keep first)
ocean_areas_df[age_col] = pd.to_numeric(ocean_areas_df[age_col], errors='coerce')
ocean_areas_df = ocean_areas_df.dropna(subset=[age_col])
ocean_areas_df = ocean_areas_df.sort_values(age_col).drop_duplicates(subset=[age_col], keep='first').reset_index(drop=True)

# Standardize to 'Age_Ma' for downstream code
ocean_areas_df = ocean_areas_df.rename(columns={age_col: 'Age_Ma'})
src = ocean_areas_df.sort_values("Age_Ma").reset_index(drop=True).copy()

# Build 0.5 Myr target grid within the original bounds (no extrapolation)
age_min, age_max = float(src["Age_Ma"].min()), float(src["Age_Ma"].max())
# snap bounds to 0.5 Myr grid inside the domain
ages_05 = np.arange(np.ceil(age_min*2)/2, np.floor(age_max*2)/2 + 0.001, 0.5)

# Interpolate numeric columns except Age_Ma via natural cubic spline
num_cols = [c for c in src.columns if c != "Age_Ma" and np.issubdtype(src[c].dtype, np.number)]
dst = pd.DataFrame({"Age_Ma": ages_05})

x = src["Age_Ma"].to_numpy(dtype=float)
for col in num_cols:
    y = src[col].to_numpy(dtype=float)
    # Natural cubic spline (second derivative = 0 at ends)
    cs = CubicSpline(x, y, bc_type="natural")
    dst[col] = cs(ages_05)

# Save and preview
from pathlib import Path
# Save and preview (ensure output directory exists)
from pathlib import Path
import pandas as pd
print('Working directory:', Path.cwd().resolve())
OUT_DIR = Path('ocean_basin_area_outputs').resolve()
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_CSV = OUT_DIR / 'ocean_basin_areas_0_70Ma_0.5my_million_km2.csv'
dst.to_csv(OUT_CSV, index=False)
print(f'Saved 0.5 Myr spline interpolation to: {OUT_CSV}')
if not OUT_CSV.exists():
    raise IOError(f'File was not created: {OUT_CSV}')
# Sanity check: reload and show a few rows
_check = pd.read_csv(OUT_CSV)
print('Saved rows/cols:', _check.shape)
_check.head()

## Plot interpolated ocean areas (0.5 Myr spline)

This cell plots the **0.5 Myr spline-interpolated** ocean-basin areas.  
It attempts to reuse your **existing color scheme** if a `BASIN_COLORS` dict is defined earlier.  
Otherwise, it falls back to sensible defaults for common basins.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# Load the interpolated CSV we just saved
try:
    OUT_CSV  # from previous cell
except NameError:
    OUT_CSV = Path('ocean_basin_area_outputs/ocean_areas_0p5Myr_interp_spline.csv')
dfi = pd.read_csv(OUT_CSV)

# Ensure proper ordering by age
dfi = dfi.sort_values('Age_Ma').reset_index(drop=True)

# Detect numeric series (basin columns)
num_cols = [c for c in dfi.columns if c != 'Age_Ma' and pd.api.types.is_numeric_dtype(dfi[c])]
if not num_cols:
    raise ValueError("No numeric basin columns found to plot aside from 'Age_Ma'.")

# Color handling: reuse prior BASIN_COLORS if provided; else set defaults
default_colors = {
    'Pacific':  '#1f77b4',
    'Atlantic': '#ff7f0e',
    'Indian':   '#2ca02c',
    'Southern': '#d62728',
    'Arctic':   '#9467bd',
}
BASIN_COLORS = globals().get('BASIN_COLORS', default_colors)

# Figure
plt.figure(figsize=(10,6), dpi=150)

for i, col in enumerate(num_cols):
    color = BASIN_COLORS.get(col, None)
    if color is None:
        # fallback to matplotlib tab10 cycle if not in dict
        color = plt.rcParams['axes.prop_cycle'].by_key().get('color', ['C0','C1','C2','C3','C4','C5','C6','C7','C8','C9'])[i % 10]
    plt.plot(dfi['Age_Ma'], dfi[col], lw=1.8, label=col, color=color)

plt.xlabel(globals().get('X_LABEL', 'Age (Ma)'))
plt.ylabel(globals().get('Y_LABEL', 'Area (million km²)'))
plt.gca().invert_xaxis()
plt.title(globals().get('TITLE_INTERP', 'Ocean-basin areas (0.5 Myr spline)'))
plt.grid(True, linestyle=':', linewidth=0.5)

# Respect prior axis direction if user defined it earlier; otherwise leave as-is
if globals().get('AGE_AXIS_REVERSED', False):
    plt.gca().invert_xaxis()

plt.legend(ncol=2, fontsize=9)
plt.tight_layout()
plt.show()
xy_path = os.path.join(OUT_DIR, "areas_timeseries_interp.png")
fig.savefig(xy_path, dpi=150); plt.show()
print("Saved:", xy_path)